# Double Descent

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt
import numpy as np

## Data Preparation

In [ ]:
# Noisy training data
X_train_np, y_train_np = make_moons(n_samples=150, noise=0.1, random_state=42)

noise_rate = 0.15
n_flip = int(noise_rate * len(y_train_np))
flip_idx = np.random.RandomState(0).choice(len(y_train_np), n_flip, replace=False)
y_train_np[flip_idx] = 1 - y_train_np[flip_idx]

# Clean test data
X_test_np, y_test_np = make_moons(n_samples=300, noise=0.1, random_state=99)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.FloatTensor(X_train_np).to(device)
y_train_t = torch.FloatTensor(y_train_np).unsqueeze(1).to(device)
X_test_t  = torch.FloatTensor(X_test_np).to(device)
y_test_t  = torch.FloatTensor(y_test_np).unsqueeze(1).to(device)

print(f"Train: {len(X_train_np)} samples, Test: {len(X_test_np)} samples, Device: {device}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_train_np[:, 0], X_train_np[:, 1], c=y_train_np, cmap='viridis', s=10)
axes[0].set_title(f"Training Data (15% labels flipped)")
axes[1].scatter(X_test_np[:, 0], X_test_np[:, 1], c=y_test_np, cmap='viridis', s=10)
axes[1].set_title("Test Data (clean labels)")
plt.tight_layout()
plt.show()

## Model

### Definition

In [ ]:
import torch.nn as nn
import torch.optim as optim

class NN(nn.Module):
    def __init__(self, num_hidden_layers=5, neurons_per_layer=16, input_size=2, output_size=1, hidden_sizes=None):
        super(NN, self).__init__()
        
        # Accept either an explicit list of hidden sizes or (num_hidden_layers, neurons_per_layer)
        if hidden_sizes is None:
            hidden_sizes = [neurons_per_layer] * num_hidden_layers

        layers = []
        in_features = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            layers.append(nn.ReLU())
            in_features = h
        # Final linear output (no activation; use BCEWithLogitsLoss)

        layers.append(nn.Linear(in_features, output_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# Vary width over a large range to cross the interpolation threshold.
# Keep depth fixed so we isolate width as the complexity axis.
depth = 2
widths = list(range(1, 21)) + [25, 32, 40, 50, 64, 80, 100, 128, 200, 300, 512]

### Training

#### Increasing Parameters

In [ ]:
# train_losses = []
# test_losses = []

train_errors = []
test_errors  = []
num_params_list = []

for width in widths:
    model = NN(num_hidden_layers=depth, neurons_per_layer=width).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Train
    num_epochs = 10000
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_tensor), y_train_tensor)
        loss.backward()
        optimizer.step()

    # model.eval()
    # with torch.no_grad():
    #     train_loss_final = criterion(model(X_train_tensor), y_train_tensor).item()
    #     test_loss_final  = criterion(model(X_test_tensor), y_test_tensor).item()

    model.eval()
    with torch.no_grad():
        train_preds = (model(X_train_tensor) > 0).float()
        train_error = (train_preds != y_train_tensor).float().mean().item()
        test_preds = (model(X_test_tensor) > 0).float()
        test_error = (test_preds != y_test_tensor).float().mean().item()
    
    n_params = sum(p.numel() for p in model.parameters())
    train_errors.append(train_error)
    test_errors.append(test_error)
    num_params_list.append(n_params)
    
    print(f"Width {width}: \tTrain Error: {train_error:.4f},  Test Error: {test_error:.4f},  Params: {n_params}")

    # train_losses.append(train_loss_final)
    # test_losses.append(test_loss_final)

    # print(f"Width {width}: \tTrain Loss: {train_loss_final:.4f}, Test Loss: {test_loss_final:.4f}, Params: {n_params}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(num_params_list, train_errors, marker='o', markersize=4, label="Train Error")
plt.plot(num_params_list, test_errors,  marker='o', markersize=4, label="Test Error")
plt.xscale('log')
plt.xlabel("Number of Parameters (log scale)")
plt.ylabel("Classification Error (0-1)")
plt.title(f"Double Descent: Depth={depth}, Width varied")
plt.axvline(x=len(X_train_tensor), color='gray', linestyle='--', alpha=0.7,
            label=f'Interpolation threshold (~{len(X_train_tensor)} params)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### Increasing Epochs

In [ ]:
# Pick a width where #params ≈ #train_samples (the interpolation threshold)
# depth=2, width=w → params = w² + 5w + 1
# w=10 → 151 params ≈ 150 samples ← right at the threshold
# w=12 → 205 params ← just above

width = 12
depth = 2
total_epochs = 50000
log_every = 50  # record error every 50 epochs

model = NN(num_hidden_layers=depth, neurons_per_layer=width).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_params = sum(p.numel() for p in model.parameters())
print(f"Width={width}, Depth={depth}, Params={n_params}, Train samples={len(X_train_np)}")

epochs_log = []
train_errors_log = []
test_errors_log = []

for epoch in range(total_epochs):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % log_every == 0:
        model.eval()
        with torch.no_grad():
            train_err = ((model(X_train_t) > 0).float() != y_train_t).float().mean().item()
            test_err  = ((model(X_test_t) > 0).float() != y_test_t).float().mean().item()

        epochs_log.append(epoch)
        train_errors_log.append(train_err)
        test_errors_log.append(test_err)

        if epoch % 5000 == 0:
            print(f"Epoch {epoch:>6d}: Train Error={train_err:.4f}, Test Error={test_err:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(epochs_log, train_errors_log, label="Train Error", alpha=0.8)
plt.plot(epochs_log, test_errors_log,  label="Test Error", alpha=0.8)
plt.xlabel("Epoch")
plt.ylabel("Classification Error (0-1)")
plt.title(f"Epoch-wise Double Descent (Width={width}, Depth={depth}, Params={n_params})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Visualization

### Train and Test Loss

In [ ]:
# Plot num_params on x-axis (log scale) — the canonical double descent x-axis
plt.figure(figsize=(10, 5))
plt.plot(num_params_list, train_losses, marker='o', label="Training Loss")
plt.plot(num_params_list, test_losses,  marker='o', label="Test Loss")
plt.xscale('log')
plt.xlabel("Number of Parameters (log scale)")
plt.ylabel("Loss")
plt.title(f"Double Descent: Depth={depth}, Width varied")
plt.axvline(x=num_params_list[train_losses.index(min(train_losses))],
            color='gray', linestyle='--', label='~Interpolation threshold')

plt.legend()
plt.show()

In [ ]:
# Secondary plot: width on x-axis (linear)
plt.figure(figsize=(10, 5))
plt.plot(widths, train_losses, marker='o', label="Training Loss")
plt.plot(widths, test_losses,  marker='o', label="Test Loss")
plt.xlabel("Width (neurons per hidden layer)")
plt.ylabel("Loss")
plt.title(f"Width vs Loss — Depth={depth}")
plt.legend()
plt.show()

### Decision Boundary